# Task 2.2 – Data Dictionary và KPI
Chạy lại từ dữ liệu raw; CSV kết quả được tạo tự động. Không chỉnh sửa raw. Định nghĩa và quy tắc được quản lý trong scripts/data_dictionary_kpi.py.

In [1]:
from pathlib import Path
import importlib.util
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'data_dictionary_kpi.py').is_file() and (p / 'data' / 'raw').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Mở notebook trong thư mục repo có data/raw.')
spec = importlib.util.spec_from_file_location('dictionary_kpi', ROOT / 'scripts' / 'data_dictionary_kpi.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
dictionary = module.build_dictionary()
module.write_kpis()
audit, source_checks, technical_results, technical_detail = module.build_quality_results(dictionary)
print('Đã tái tạo dictionary và kiểm tra từ 4 nguồn raw.')

[INFO] Snapshot metadata: {'listings.csv.gz': '2026-09-23', 'calendar.csv.gz': '2026-09-23', 'reviews.csv.gz': '2026-09-23', 'neighbourhoods.csv': '2026-09-23'}


Đã tái tạo dictionary và kiểm tra từ 4 nguồn raw.


## Schema và quy tắc chuyển đổi

In [2]:
print('Số trường:', len(dictionary))
display(dictionary.groupby('source_name').size().rename('column_count').to_frame())
display(dictionary[['source_name', 'column_name', 'business_meaning', 'proposed_transformation']])

Số trường: 103


,column_count
source_name,
Calendar,5
Listings,90
Neighbourhoods,2
Reviews,6


,source_name,column_name,business_meaning,proposed_transformation
0,Listings,id,Mã định danh duy nhất của tin đăng Airbnb.,"Lưu BIGINT; giữ chính xác chữ số, không chuyển..."
1,Listings,listing_url,Đường dẫn trang tin đăng trên Airbnb.,Lưu VARCHAR/TEXT; chuẩn hóa khoảng trắng và NU...
2,Listings,scrape_id,Mã lần thu thập dữ liệu của nguồn.,"Lưu BIGINT; giữ chính xác chữ số, không chuyển..."
3,Listings,last_scraped,Ngày tin đăng được thu thập lần cuối.,Chuẩn hóa thành DATE; giữ NULL; ghi nhận giá t...
4,Listings,source,Nguồn hoặc trạng thái xuất xứ của bản ghi theo...,Lưu VARCHAR/TEXT; chuẩn hóa khoảng trắng và NU...
...,...,...,...,...
98,Reviews,reviewer_id,Mã định danh của người viết đánh giá.,"Lưu BIGINT; giữ chính xác chữ số, không chuyển..."
99,Reviews,reviewer_name,Tên hiển thị của người viết đánh giá.,Lưu VARCHAR/TEXT; chuẩn hóa khoảng trắng và NU...
100,Reviews,comments,Nội dung văn bản đánh giá do người dùng gửi.,Giữ TEXT và NULL; không suy ra điểm review từ ...
101,Neighbourhoods,neighbourhood_group,Nhóm hoặc vùng địa lý chứa khu vực.,Lưu VARCHAR/TEXT; chuẩn hóa khoảng trắng và NU...


## Định nghĩa: 5 Business KPI, 2 chỉ tiêu hỗ trợ, 5 Technical KPI

In [3]:
display(module.pd.DataFrame(module.BUSINESS_KPIS))
display(module.pd.DataFrame(module.TECH_KPIS))

,kpi_id,kpi_name,business_definition,source_fields,formula,filters,numerator,denominator,unit,grain,interpretation,caveats
0,KPI-01,Số listing duy nhất theo snapshot,Số lượng mã listing phân biệt được ghi nhận tr...,Listings.id; Listings.scrape_id; Listings.last...,COUNT(DISTINCT id),Chỉ dùng một snapshot; loại id NULL; xác nhận ...,Số mã listing duy nhất hợp lệ.,Không áp dụng.,Listing,Một listing tại snapshot,"Quy mô tin đăng được ghi nhận tại snapshot, kh...",Không đếm dòng Calendar/Reviews; không gộp nhi...
1,KPI-02,Trung vị giá niêm yết mỗi đêm,Trung vị giá niêm yết mỗi đêm của các listing ...,Listings.id; Listings.price; Listings.price_qu...,"MEDIAN(price_clean), mỗi listing một lần; pric...","Một snapshot; price không NULL, parse được, hữ...",Không áp dụng cho median.,Không áp dụng; báo cáo coverage = listing có g...,SGD/đêm,Một listing tại snapshot,Một nửa listing có giá hợp lệ có giá không lớn...,Trong dữ liệu hiện tại có 2.855 báo giá ghi rõ...
2,KPI-03,Tỷ trọng loại phòng,Tỷ lệ listing thuộc từng room_type trên tổng l...,Listings.id; Listings.room_type; Listings.neig...,Số listing duy nhất thuộc room_type X / tổng l...,Giữ bộ lọc snapshot và khu vực; khi tính tỷ tr...,Số listing duy nhất thuộc room_type X trong ph...,Tổng listing duy nhất có room_type hợp lệ tron...,% listing,Một listing tại snapshot,Mô tả cơ cấu loại phòng trong tổng tin đăng có...,Không dùng số dòng Calendar/Reviews làm mẫu số...
3,KPI-04,Tỷ lệ bản ghi lịch còn available,Tỷ lệ bản ghi listing-ngày có trạng thái avail...,Calendar.listing_id; Calendar.date; Calendar.a...,SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 E...,listing_id và date hợp lệ; available chuẩn hóa...,Số dòng listing-ngày hợp lệ có available = TRUE.,Số dòng listing-ngày hợp lệ có available = TRU...,% bản ghi listing-ngày,Một listing tại một ngày lịch và một snapshot,Tỷ lệ trạng thái lịch được đánh dấu còn khả dụ...,"FALSE chỉ là không khả dụng, không chứng minh ..."
4,KPI-05,Điểm review tổng thể trung bình,Điểm review tổng thể trung bình giữa các listi...,Listings.id; Listings.review_scores_rating,AVG(review_scores_rating) trên mỗi listing duy...,Một snapshot; điểm là số hữu hạn trong miền 1–...,Tổng điểm review tổng thể của các listing có đ...,Số listing duy nhất có điểm review tổng thể hợ...,Điểm (thang quan sát 1–5),Một listing tại snapshot,Điểm tổng thể trung bình giữa các listing có đ...,Dữ liệu snapshot hiện tại quan sát được miền 1...
5,SUP-01,Trung vị số review trên mỗi listing,Trung vị số review được ghi nhận trên mỗi list...,Listings.id; Listings.number_of_reviews,"MEDIAN(number_of_reviews), mỗi listing một lần.",Một snapshot; number_of_reviews là số hợp lệ v...,Không áp dụng.,Không áp dụng.,Review/listing,Một listing tại snapshot,Mức số lượng review điển hình của listing tron...,Không thay NULL bằng 0; không mặc định số dòng...
6,SUP-02,Số bản ghi review trong kỳ,Số bản ghi review hợp lệ được ghi nhận trong k...,Reviews.listing_id; Reviews.id; Reviews.date,"Đếm bản ghi theo khóa (listing_id, id) sau kiể...",Ngày review hợp lệ và nằm trong kỳ; listing_id...,Số bản ghi review hợp lệ trong kỳ.,Không áp dụng.,Bản ghi review,Một bản ghi review,"Lượng review quan sát được trong kỳ, có thể ph...",Không suy ra số booking; không coi review date...


,technical_kpi_id,technical_kpi_name,definition,formula,scope,unit,grain,interpretation,caveats
0,TQ-01,Tỷ lệ thiếu dữ liệu (Missing Value Rate),Tỷ lệ ô NULL/thiếu trên tổng số ô thuộc danh s...,Tổng số ô NULL ở các trường bắt buộc / tổng số...,Báo cáo theo từng trường và tổng hợp theo nguồ...,%,Một trường dữ liệu trong một nguồn/lớp và snap...,Tỷ lệ càng thấp cho thấy mức độ đầy đủ càng ca...,Không gộp mọi cột tùy ý; phân biệt NULL hợp lệ...
1,TQ-02,Số bản ghi trùng khóa (Duplicate Key Count),Số bản ghi vượt quá một bản ghi duy nhất theo ...,COUNT(rows) - COUNT(DISTINCT business_key) hoặ...,Kiểm tra RAW hiện tại (một snapshot): Listings...,Bản ghi trùng,Theo khóa/grain của từng nguồn/lớp,Giá trị 0 nghĩa là không phát hiện bản ghi trù...,Không mặc định Reviews.id riêng là duy nhất; p...
2,TQ-03,Tỷ lệ toàn vẹn tham chiếu (Referential Integri...,Tỷ lệ bản ghi con có khóa tham chiếu tồn tại t...,Số bản ghi con có khóa cha hợp lệ / tổng bản g...,Calendar.listing_id và Reviews.listing_id tham...,%,Bản ghi cần liên kết của từng quan hệ,Tỷ lệ càng cao cho thấy liên kết giữa các nguồ...,Báo cáo riêng số khóa NULL và số khóa mồ côi; ...
3,TQ-04,Tỷ lệ bản ghi hợp lệ (Valid Record Rate),Tỷ lệ bản ghi đáp ứng toàn bộ quy tắc chất lượ...,Số bản ghi đạt tất cả kiểm tra bắt buộc / tổng...,"Kiểm tra kiểu dữ liệu, miền giá trị, trường bắ...",%,Một bản ghi tại từng nguồn/lớp,Đo tỷ lệ dữ liệu đáp ứng tiêu chí sử dụng sau ...,Công bố rõ rule và số dòng bị loại/sửa; không ...
4,TQ-05,Tính nhất quán khi chạy lại pipeline (Pipeline...,Mức độ giống nhau của số lượng bản ghi và KPI ...,So sánh row count và KPI sau lần chạy lại với ...,Thực hiện tối thiểu hai lần chạy cùng snapshot...,PASS/FAIL và chênh lệch bản ghi/KPI,Một lần chạy pipeline cho một snapshot,PASS chứng minh pipeline có thể tái hiện kết q...,Không thể tính từ file raw đơn lẻ; phải chạy p...


## Bằng chứng từ raw và Technical KPI tối thiểu

In [4]:
display(source_checks)
display(technical_results)
display(technical_detail)
assert len(dictionary) == 103
assert technical_results.loc[technical_results.technical_kpi_id == 'TQ-05', 'trang_thai'].iloc[0] == 'CHƯA ĐÁNH GIÁ'

,check_id,noi_dung,so_ban_ghi_kiem_tra,ket_qua,trang_thai
0,SRC-01,Currency trong price_quote_raw.quote.currency,2890,"JSON báo giá có dữ liệu: 2,890; SGD: 2,855; JS...",Chỉ xác định SGD khi currency trong JSON bằng SGD
1,SRC-02,Miền giá trị review_scores_rating,1839,"Min: 1.0; Max: 5.0; Hợp lệ 1–5: 1,839; Ngoài m...",NULL không được tự động xem là điểm 0
2,SRC-03,Đối chiếu Listings.price với price_quote_price...,2725,"Có đủ hai giá trị để đối chiếu: 2,725; Khớp tr...",Chỉ đối chiếu các listing có currency SGD và đ...


,technical_kpi_id,technical_kpi_name,ket_qua,don_vi,trang_thai,dien_giai
0,TQ-01,Tỷ lệ thiếu dữ liệu,0.0,%,Đã tính,"0/3,657,376 ô ở các trường bắt buộc bị thiếu."
1,TQ-02,Số bản ghi trùng khóa,0.0,Dòng trùng dư,Đã tính,Tổng số dòng trùng dư theo khóa/grain đã xác đ...
2,TQ-03,Tỷ lệ toàn vẹn tham chiếu,100.0,%,Đã tính,"1,220,158/1,220,158 khóa con có giá trị liên k..."
3,TQ-04,Tỷ lệ bản ghi hợp lệ,100.0,%,Đã tính theo luật tối thiểu công bố,"1,220,213/1,220,213 dòng đạt luật kiểm tra tối..."
4,TQ-05,Tính nhất quán khi chạy lại pipeline,NaN,PASS/FAIL,CHƯA ĐÁNH GIÁ,Cần log của ít nhất hai lần chạy ETL với cùng ...


,technical_kpi_id,nguon,truong_kiem_tra,so_dong,so_o_kiem_tra,so_o_thieu,ket_qua_percent,ghi_chu,khoa_grain,so_dong_nguon,...,so_dong_thieu_thanh_phan_khoa,quan_he,tong_dong_con,khoa_con_null_or_blank,khoa_con_co_gia_tri,khoa_mo_coi,khoa_hop_le,quy_tac_hop_le,so_dong_hop_le,so_dong_khong_hop_le
0,TQ-01,Listings,"id, scrape_id",3208.0,6416.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TQ-01,Calendar,"listing_id, date, available",1170950.0,3512850.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TQ-01,Reviews,"listing_id, id, date",46000.0,138000.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TQ-01,Neighbourhoods,"neighbourhood_group, neighbourhood",55.0,110.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TQ-01,Tổng hợp,Các trường bắt buộc của 4 nguồn,NaN,3657376.0,0.0,0.0,Tỷ lệ tính trên tổng số ô thuộc các trường bắt...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,TQ-02,Listings,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",id,3208.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,TQ-02,Calendar,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",listing_id + date,1170950.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,TQ-02,Reviews,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",listing_id + id,46000.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,TQ-02,Neighbourhoods,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",neighbourhood_group + neighbourhood,55.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,TQ-03,NaN,NaN,NaN,NaN,NaN,100.0,Tỷ lệ tính trên khóa con có giá trị. NULL/rỗng...,NaN,NaN,...,NaN,Calendar.listing_id -> Listings.id,1170950.0,0.0,1170950.0,0.0,1170950.0,NaN,NaN,NaN


## Giới hạn
TQ-01 đến TQ-04 ở đây là kiểm tra raw theo luật tối thiểu công bố, chưa chứng minh chất lượng Gold. TQ-05 cần hai lần chạy ETL thực tế. Miền rating 1–5 là miền quan sát; cần xác nhận metadata nguồn khi chốt ETL. Các file outputs/task_2_2 được tạo lại khi chạy, không phải dữ liệu đầu vào.